# YingAdaptAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.YingAdaptAge)

class YingAdaptAge(pyagingModel):
    def __init__(self):
        super().__init__()

    def preprocess(self, x):
        return x

    def postprocess(self, x):
        return x



In [3]:
model = pya.models.YingAdaptAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "yingadaptage"
model.metadata["data_type"] = "DNA methylation"  # Paper: All three clocks use whole-blood DNA-methylation beta values.
model.metadata["species"] = "Homo sapiens"  # Paper: The models were trained in human Generation Scotland blood samples.
model.metadata["year"] = 2024
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Ying, K., Liu, H., Tarkhov, A.E. et al. Causality-enriched epigenetic age uncouples damage and adaptation. Nature Aging 4, 231–246 (2024)."
model.metadata["doi"] = "https://doi.org/10.1038/s43587-023-00557-0"
model.metadata["notes"] = "Causality-enriched age predictor restricted to adaptive/protective age-related CpGs, with feature penalties weighted by EWMR causality scores."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: The model was trained in whole-blood methylation.
model.metadata["predicts"] = ["adaptive epigenetic age"]  # Paper: AdaptAge is designed to track protective adaptations accumulated during aging.
model.metadata["training_target"] = ["chronological age"]  # Paper: All three causality-enriched elastic-net models were trained to predict chronological age.
model.metadata["unit"] = ["years"]  # Paper: Model selection used mean absolute error in years against chronological age.
model.metadata["model_type"] = "causality-weighted elastic net regression"  # Paper: Feature-specific elastic-net penalty factors were assigned from each CpG's causality score.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: Training used Generation Scotland whole-blood methylation at CpGs available to the 450K-based analysis.
model.metadata["population"] = "adults"  # Paper: Age-related methylation was estimated in 7,036 Generation Scotland participants aged 18–93, and 2,664 blood samples were used for clock training.
model.metadata["journal"] = "Nature Aging"
model.metadata["last_author"] = "Vadim N. Gladyshev"
model.metadata["n_features"] = 999
model.metadata["citations"] = 183
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

#### Download directly with curl

In [5]:
supplementary_url = "https://static-content.springer.com/esm/art%3A10.1038%2Fs43587-023-00557-0/MediaObjects/43587_2023_557_MOESM6_ESM.zip"
supplementary_file_name = "43587_2023_557_MOESM6_ESM.zip"
os.system(f"curl -o {supplementary_file_name} {supplementary_url}")
os.system(f'unzip {supplementary_file_name}')

0

## Load features

#### From CSV file

In [6]:
df = pd.read_csv('YingAdaptAge.csv')
df['feature'] = df['term']
df['coefficient'] = df['estimate']
model.features = df['feature'][1:].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(df['coefficient'][1:].tolist()).unsqueeze(0)
intercept = torch.tensor([df['coefficient'][0]])

#### Linear model

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': ('Ying, Kejun, et al. "Causality-enriched epigenetic age '
              'uncouples damage and adaptation." Nature Aging (2024): 1-16.',),
 'clock_name': 'yingadaptage',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1038/s43587-023-00557-0',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2024}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00008671', 'cg00017970', 'cg00048759', 'cg00050402', 'cg00089550', 'cg00099240', 'cg00108164', 'cg00131893', 'cg00158122', 'cg00223715', 'cg00229508', 'cg00277334', 'cg00290758', 'cg00295744', 'cg00316485', 'cg00335735', 'cg00342891', 'cg00344422', 'cg00346145', 'cg00388262', 'cg00492070', 'cg00505045', 'cg00513984', 'cg00539

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: 43587_2023_557_MOESM6_ESM.zip
Deleted file: YingCausAge.csv
Deleted file: YingDamAge.csv
Deleted file: YingAdaptAge.csv
